Cell 1 — Check environment

In [ ]:
import tensorflow as tf

print("TensorFlow:", tf.__version__)
print("GPU:", tf.config.list_physical_devices("GPU"))

Cell 2 — Setup project root

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd()

if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

print("Project root:", PROJECT_ROOT)

Cell 3 — Load dataset for MobileNetV2 baseline

In [ ]:
import tensorflow as tf
from pathlib import Path

TRAIN_DIR = PROJECT_ROOT / "data/processed/train"
VAL_DIR = PROJECT_ROOT / "data/processed/val"
TEST_DIR = PROJECT_ROOT / "data/processed/test"

IMG_SIZE = (96, 96)
BATCH_SIZE = 32
SEED = 42
AUTOTUNE = tf.data.AUTOTUNE

train_ds = tf.keras.utils.image_dataset_from_directory(
    TRAIN_DIR,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode="int",
    shuffle=True,
    seed=SEED,
)

val_ds = tf.keras.utils.image_dataset_from_directory(
    VAL_DIR,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode="int",
    shuffle=False,
)

test_ds = tf.keras.utils.image_dataset_from_directory(
    TEST_DIR,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode="int",
    shuffle=False,
)

class_names = train_ds.class_names
num_classes = len(class_names)

train_ds = train_ds.prefetch(AUTOTUNE)
val_ds = val_ds.prefetch(AUTOTUNE)
test_ds = test_ds.prefetch(AUTOTUNE)

print("Number of classes:", num_classes)
print("Class names:", class_names[:10])

Cell 4 — Compute class weight

In [ ]:
import numpy as np
from sklearn.utils.class_weight import compute_class_weight

train_counts = {}

for class_dir in sorted(TRAIN_DIR.iterdir(), key=lambda p: int(p.name)):
    if class_dir.is_dir():
        train_counts[int(class_dir.name)] = len(list(class_dir.glob("*.*")))

all_train_labels = []

for class_id, count in train_counts.items():
    all_train_labels.extend([class_id] * count)

all_train_labels = np.array(all_train_labels)

class_weights_array = compute_class_weight(
    class_weight="balanced",
    classes=np.unique(all_train_labels),
    y=all_train_labels,
)

class_weight = {
    int(class_id): float(weight)
    for class_id, weight in zip(np.unique(all_train_labels), class_weights_array)
}

print("Number of class weights:", len(class_weight))
print("Example:", list(class_weight.items())[:5])

Cell 5 — Build and train MobileNetV2 baseline

In [ ]:
from tensorflow.keras import layers, models
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau
import matplotlib.pyplot as plt

MODELS_DIR = PROJECT_ROOT / "models"
FIGURES_DIR = PROJECT_ROOT / "results/figures"
REPORTS_DIR = PROJECT_ROOT / "results/reports"

MODELS_DIR.mkdir(parents=True, exist_ok=True)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)
REPORTS_DIR.mkdir(parents=True, exist_ok=True)

MOBILENET_MODEL_PATH = MODELS_DIR / "mobilenetv2_baseline.keras"
MOBILENET_HISTORY_PATH = FIGURES_DIR / "mobilenetv2_accuracy_loss.png"

base_model = MobileNetV2(
    input_shape=(96, 96, 3),
    include_top=False,
    weights="imagenet",
)

base_model.trainable = False

data_augmentation = tf.keras.Sequential([
    layers.RandomRotation(0.05),
    layers.RandomZoom(0.1),
    layers.RandomTranslation(0.05, 0.05),
], name="data_augmentation")

inputs = layers.Input(shape=(96, 96, 3))
x = data_augmentation(inputs)
x = tf.keras.applications.mobilenet_v2.preprocess_input(x)
x = base_model(x, training=False)
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dropout(0.3)(x)
outputs = layers.Dense(num_classes, activation="softmax")(x)

mobilenet_model = models.Model(inputs, outputs)

mobilenet_model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"],
)

mobilenet_model.summary()

callbacks = [
    EarlyStopping(
        monitor="val_accuracy",
        patience=5,
        restore_best_weights=True,
    ),
    ModelCheckpoint(
        MOBILENET_MODEL_PATH,
        monitor="val_accuracy",
        save_best_only=True,
    ),
    ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.5,
        patience=2,
        min_lr=1e-6,
    ),
]

train_ds_for_week2 = train_ds.take(700)
val_ds_for_week2 = val_ds.take(150)

EPOCHS = 20

mobilenet_history = mobilenet_model.fit(
    train_ds_for_week2,
    validation_data=val_ds_for_week2,
    epochs=EPOCHS,
    callbacks=callbacks,
    class_weight=class_weight,
)

Cell 6 — Plot MobileNetV2 training history

In [ ]:
history = mobilenet_history.history

plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
plt.plot(history["accuracy"], label="Training Accuracy")
plt.plot(history["val_accuracy"], label="Validation Accuracy")
plt.title("MobileNetV2 Baseline Accuracy")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(history["loss"], label="Training Loss")
plt.plot(history["val_loss"], label="Validation Loss")
plt.title("MobileNetV2 Baseline Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend()

plt.tight_layout()
plt.savefig(MOBILENET_HISTORY_PATH, dpi=300)
plt.show()

print("Saved:", MOBILENET_HISTORY_PATH)
print("Model exists:", MOBILENET_MODEL_PATH.exists())

Cell 7 — Evaluate MobileNetV2 baseline

In [ ]:
best_mobilenet_model = tf.keras.models.load_model(MOBILENET_MODEL_PATH)

test_loss, test_accuracy = best_mobilenet_model.evaluate(test_ds)

print("MobileNetV2 test loss:", test_loss)
print("MobileNetV2 test accuracy:", test_accuracy)
print("MobileNetV2 test accuracy percent:", test_accuracy * 100)

Cell 8 — Save MobileNetV2 report and confusion matrix

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from sklearn.metrics import (
    accuracy_score,
    f1_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
)

y_true = []
y_pred = []

for images, labels in test_ds:
    predictions = best_mobilenet_model.predict(images, verbose=0)
    predicted_labels = np.argmax(predictions, axis=1)

    y_true.extend(labels.numpy())
    y_pred.extend(predicted_labels)

y_true = np.array(y_true)
y_pred = np.array(y_pred)

accuracy = accuracy_score(y_true, y_pred)
macro_f1 = f1_score(y_true, y_pred, average="macro")
weighted_f1 = f1_score(y_true, y_pred, average="weighted")

print("MobileNetV2 Accuracy:", accuracy)
print("MobileNetV2 Macro-F1:", macro_f1)
print("MobileNetV2 Weighted-F1:", weighted_f1)

report = classification_report(y_true, y_pred, digits=4, zero_division=0)

REPORT_PATH = REPORTS_DIR / "mobilenetv2_classification_report.txt"

with open(REPORT_PATH, "w", encoding="utf-8") as f:
    f.write("MobileNetV2 Baseline Classification Report\n")
    f.write("==========================================\n\n")
    f.write(f"Test loss: {test_loss:.4f}\n")
    f.write(f"Test accuracy: {test_accuracy:.4f}\n")
    f.write(f"Accuracy: {accuracy:.4f}\n")
    f.write(f"Macro-F1: {macro_f1:.4f}\n")
    f.write(f"Weighted-F1: {weighted_f1:.4f}\n\n")
    f.write(report)

cm = confusion_matrix(y_true, y_pred)

fig, ax = plt.subplots(figsize=(12, 10))
disp = ConfusionMatrixDisplay(confusion_matrix=cm)
disp.plot(ax=ax, values_format="d", xticks_rotation=90)
plt.title("MobileNetV2 Baseline Confusion Matrix")
plt.tight_layout()

CM_PATH = FIGURES_DIR / "mobilenetv2_confusion_matrix.png"
plt.savefig(CM_PATH, dpi=300)
plt.show()

print("Saved report:", REPORT_PATH)
print("Saved confusion matrix:", CM_PATH)